# 07. GCS Raw (Glasgow Coma Scale 원본 추출)

## 목적
코호트 환자의 GCS 점수를 **1시간 단위**로 추출

## 입력
- `cohort_base.csv`: 기본 코호트

## 출력
- `gcs_raw.csv`: 시간별 GCS 점수 (1 row = 1 patient × 1 hour)

## Item IDs
| Item ID | 항목 | 범위 |
|---------|------|------|
| 220739 | GCS - Eye Opening | 1-4 |
| 223900 | GCS - Verbal Response | 1-5 |
| 223901 | GCS - Motor Response | 1-6 |

GCS Total = Eye + Verbal + Motor (3-15)

In [ ]:
import duckdb
import pandas as pd
import os

DB_PATH = '../data/duckdb/mimic_total.duckdb'
INPUT_DIR = '../data/processed'
OUTPUT_DIR = '../data/processed'

con = duckdb.connect(DB_PATH)
print("=== 07. GCS Raw 추출 시작 ===")

## Step 1: 코호트 로드

In [ ]:
print("Step 1: 코호트 로드")

cohort_path = os.path.join(INPUT_DIR, 'cohort_base.csv')
df_cohort = pd.read_csv(cohort_path, parse_dates=['intime', 'outtime'])

print(f"✓ 코호트 로드 완료: {len(df_cohort):,}명")

con.register('cohort', df_cohort)

## Step 2: GCS 데이터 추출

In [ ]:
print("\nStep 2: GCS 데이터 추출 (1시간 단위)")

gcs_query = """
WITH gcs_values AS (
    SELECT
        c.stay_id,
        date_trunc('hour', CAST(ce.charttime AS TIMESTAMP)) as charttime_h,
        ce.itemid,
        TRY_CAST(ce.valuenum AS DOUBLE) as val_num,
        CAST(ce.charttime AS TIMESTAMP) as charttime_ts
    FROM cohort c
    INNER JOIN chartevents ce ON c.stay_id = ce.stay_id
    WHERE ce.itemid IN ('220739', '223900', '223901')
      AND TRY_CAST(ce.valuenum AS DOUBLE) IS NOT NULL
      AND CAST(ce.charttime AS TIMESTAMP) >= c.intime
      AND CAST(ce.charttime AS TIMESTAMP) <= c.outtime
),
aggregated AS (
    SELECT
        stay_id,
        charttime_h,
        MAX_BY(val_num, charttime_ts) FILTER (WHERE itemid = '220739') as gcs_eye,
        MAX_BY(val_num, charttime_ts) FILTER (WHERE itemid = '223900') as gcs_verbal,
        MAX_BY(val_num, charttime_ts) FILTER (WHERE itemid = '223901') as gcs_motor,
        MIN(CASE WHEN itemid = '220739' THEN val_num END) as gcs_eye_min,
        MIN(CASE WHEN itemid = '223900' THEN val_num END) as gcs_verbal_min,
        MIN(CASE WHEN itemid = '223901' THEN val_num END) as gcs_motor_min
    FROM gcs_values
    GROUP BY stay_id, charttime_h
)
SELECT
    stay_id,
    charttime_h,
    gcs_eye,
    gcs_verbal,
    gcs_motor,
    COALESCE(gcs_eye, 0) + COALESCE(gcs_verbal, 0) + COALESCE(gcs_motor, 0) as gcs_total,
    COALESCE(gcs_eye_min, 0) + COALESCE(gcs_verbal_min, 0) + COALESCE(gcs_motor_min, 0) as gcs_total_min
FROM aggregated
WHERE gcs_eye IS NOT NULL OR gcs_verbal IS NOT NULL OR gcs_motor IS NOT NULL
ORDER BY stay_id, charttime_h
"""

print("  쿼리 실행 중...")
df_gcs = con.execute(gcs_query).df()

print(f"✓ GCS 추출 완료")
print(f"  - 총 행 수: {len(df_gcs):,}개")
print(f"  - 고유 환자: {df_gcs['stay_id'].nunique():,}명")

## Step 3: 데이터 품질 확인

In [ ]:
print("\nStep 3: 데이터 품질 확인")

print("\n=== 결측치 비율 ===")
for col in ['gcs_eye', 'gcs_verbal', 'gcs_motor', 'gcs_total']:
    missing_rate = df_gcs[col].isna().mean() * 100
    print(f"  {col}: {missing_rate:.1f}% 결측")

print("\n=== GCS Total 분포 ===")
print(f"  - 평균: {df_gcs['gcs_total'].mean():.1f}")
print(f"  - 중앙값: {df_gcs['gcs_total'].median():.1f}")
print(f"  - 범위: {df_gcs['gcs_total'].min():.0f} ~ {df_gcs['gcs_total'].max():.0f}")

## Step 4: 저장

In [ ]:
print("\n" + "="*60)
print("CSV 저장")
print("="*60)

output_path = os.path.join(OUTPUT_DIR, 'gcs_raw.csv')
df_gcs.to_csv(output_path, index=False)

file_size = os.path.getsize(output_path) / (1024 * 1024)

print(f"\n✓ 저장 완료: gcs_raw.csv")
print(f"  - 파일 크기: {file_size:.2f} MB")
print(f"  - 행 수: {len(df_gcs):,}개")
print(f"  - 경로: {output_path}")

In [ ]:
print("\n=== 샘플 데이터 ===")
df_gcs.head(10)

In [ ]:
con.close()
print("\n=== 07. GCS Raw 추출 완료 ===")